In [77]:
!pip install ultralytics --quiet
!pip install torchvision --quiet
!pip install torch --quiet
!pip install pillow --quiet
!pip install scikit-learn --quiet

In [78]:
from PIL import Image
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, Subset, DataLoader
from pathlib import Path

import torchvision.transforms.v2 as transforms_v2
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [79]:
BASE_DIR = Path.cwd().parents[0]
CSV_FN = 'manifest.csv'
CSV_PATH = BASE_DIR / 'CVModel-Data_Preparation' / 'Labeling_Preprocessing' / CSV_FN

In [80]:
class CSVDataset(Dataset):

    def __init__(self, csv_path, transform=None):
        self.data = pd.read_csv(csv_path)
        self.transform = transform

        # label: string to number
        self.label_map = {label: idx for idx, label in enumerate(self.data['label'].unique())}
        self.data['label'] = self.data['label'].map(self.label_map)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['image']
        label = self.data.iloc[idx]['label']

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label
    

In [81]:
train_transform = transforms_v2.Compose([
    transforms_v2.RandomResizedCrop(224),
    transforms_v2.RandomHorizontalFlip(),
    transforms_v2.ColorJitter(0.15),
    transforms_v2.ToTensor(),
])

val_transform = transforms_v2.Compose([
    transforms_v2.Resize(256),
    transforms_v2.CenterCrop(224),
    transforms_v2.ToTensor(),
])

/opt/anaconda3/envs/sscvmodel-fine_tuning/lib/python3.12/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [82]:
full_dataset_train = CSVDataset(CSV_PATH, transform=train_transform)
full_dataset_val = CSVDataset(CSV_PATH, transform=val_transform)

In [83]:
indices = np.arange(len(full_dataset_train)) # they are same size
labels = full_dataset_train.data['label']

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

In [84]:
train_dataset = Subset(full_dataset_train, train_indices)
val_dataset = Subset(full_dataset_val, val_indices)

In [85]:
len(train_dataset), len(val_dataset)

(1219, 305)

In [86]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=True)

In [87]:
next(iter(train_loader))

[tensor([[[[0.6706, 0.6706, 0.6784,  ..., 0.8627, 0.8667, 0.8667],
           [0.6784, 0.6784, 0.6824,  ..., 0.8549, 0.8588, 0.8588],
           [0.6824, 0.6863, 0.6863,  ..., 0.8549, 0.8549, 0.8549],
           ...,
           [0.2902, 0.2824, 0.2863,  ..., 0.2784, 0.2471, 0.2510],
           [0.3020, 0.2980, 0.2706,  ..., 0.2118, 0.2235, 0.1961],
           [0.2784, 0.2667, 0.2549,  ..., 0.2431, 0.2588, 0.2431]],
 
          [[0.7255, 0.7255, 0.7333,  ..., 0.8824, 0.8863, 0.8863],
           [0.7333, 0.7333, 0.7373,  ..., 0.8745, 0.8784, 0.8784],
           [0.7373, 0.7412, 0.7412,  ..., 0.8745, 0.8745, 0.8745],
           ...,
           [0.4196, 0.4078, 0.4157,  ..., 0.3922, 0.3608, 0.3647],
           [0.4235, 0.4196, 0.3882,  ..., 0.3098, 0.3176, 0.2902],
           [0.3922, 0.3843, 0.3686,  ..., 0.3373, 0.3529, 0.3373]],
 
          [[0.8902, 0.8902, 0.8980,  ..., 0.9569, 0.9608, 0.9608],
           [0.8980, 0.8980, 0.9020,  ..., 0.9490, 0.9529, 0.9529],
           [0.9020, 0.90

In [88]:
yolo = YOLO('yolov8n-cls.pt')  
model = yolo.model 

In [89]:
num_classes = len(full_dataset_train.data['label'].unique())
model.model[-1].linear = nn.Linear(model.model[-1].linear.in_features, num_classes)

In [90]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps')
model.to(device)

ClassificationModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
   

In [91]:
epochs = 100

In [92]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs
)

In [93]:
WEIGHTS_DIR = Path.cwd().parent / 'src'
WEIGHTS_PATH = WEIGHTS_DIR / 'best_model.pt'

In [94]:
import pprint

In [95]:
idx_to_label = {idx: label for label, idx in full_dataset_train.label_map.items()}
pprint.pprint(idx_to_label)

best_val_acc = 0
patience = 15
patience_counter = 0
early_stopped = False

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    # ---- VALIDATE ----
    model.eval()
    correct = 0
    total = 0
    misclassified_pairs = {}

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            if isinstance(outputs, tuple):
                outputs = outputs[0]

            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            for true_idx, pred_idx in zip(labels.cpu().tolist(), preds.cpu().tolist()):
                if true_idx != pred_idx:
                    key = (true_idx, pred_idx)
                    misclassified_pairs[key] = misclassified_pairs.get(key, 0) + 1

    val_acc = correct / total
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), WEIGHTS_PATH)
    else:
        patience_counter += 1

    scheduler.step()
    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Accuracy: {val_acc:.4f}")

    if patience_counter >= patience:
        print('Early stopping triggered')
        early_stopped = True
        break
    
if misclassified_pairs:
    reason = "at early stopping" if early_stopped else "on final epoch"
    print(f'Misclassified classes (true -> predicted) {reason}:')
    for (true_idx, pred_idx), count in sorted(misclassified_pairs.items(), key=lambda x: x[1], reverse=True):
        true_label = idx_to_label.get(true_idx, str(true_idx))
        pred_label = idx_to_label.get(pred_idx, str(pred_idx))
        print(f"  {true_label} -> {pred_label}: {count}")
else:
    print('No misclassifications in validation on final epoch.')


{0: 'baseball',
 1: 'soccer',
 2: 'pingpong',
 3: 'basketball',
 4: 'petanque',
 5: 'volleyball',
 6: 'tennis'}

Epoch 1
Train Loss: 1.6772
Val Accuracy: 0.6426

Epoch 2
Train Loss: 1.2944
Val Accuracy: 0.7082

Epoch 3
Train Loss: 1.1572
Val Accuracy: 0.7246

Epoch 4
Train Loss: 1.0730
Val Accuracy: 0.7344

Epoch 5
Train Loss: 1.0303
Val Accuracy: 0.7344

Epoch 6
Train Loss: 0.9878
Val Accuracy: 0.7213

Epoch 7
Train Loss: 0.9664
Val Accuracy: 0.7279

Epoch 8
Train Loss: 0.9314
Val Accuracy: 0.7574

Epoch 9
Train Loss: 0.9437
Val Accuracy: 0.7475

Epoch 10
Train Loss: 0.9095
Val Accuracy: 0.7541

Epoch 11
Train Loss: 0.8856
Val Accuracy: 0.7508

Epoch 12
Train Loss: 0.8844
Val Accuracy: 0.7410

Epoch 13
Train Loss: 0.8480
Val Accuracy: 0.7311

Epoch 14
Train Loss: 0.8160
Val Accuracy: 0.7475

Epoch 15
Train Loss: 0.8299
Val Accuracy: 0.7574

Epoch 16
Train Loss: 0.8292
Val Accuracy: 0.7803

Epoch 17
Train Loss: 0.7926
Val Accuracy: 0.7770

Epoch 18
Train Loss: 0.7931
Val Accuracy: 0.76